# Cross-resolution pretrains: layer-by-layer on a random 96 × 96 crop

Run the setup cell first, then any of the five test cells (D, S, X, U-1, U-2). Each test cell draws a random 96 × 96 crop from a **monitor tile**, a tile no run ever trains on, and prints which tile and crop it drew. Set `SEED` in the setup cell to repeat a crop.

Each figure has one row per layer and three columns:
- **Left:** the ground truth for that layer.
  - D, X, U: the co-registered 2.4 µm scan, intensity-matched to the 9.36 µm input.
  - S: the real 9.36 µm slice.
- **Middle:** the model's layers interleaved with the real 9.36 µm slices.
  - D, X, U: each real slice spans 4 target layers and is shown at the second of them (green frame, `REAL`). The other layers are predictions (`pred`).
  - S: layers with visible data have a green frame. Fully hidden slices have a red frame and are 100% predicted.
- **Right:** the overlay, red = middle, green = truth, yellow = agreement.

| test | layers | checkpoint |
|---|---|---|
| D | 32 at 96 × 96 | `models/mae_crossres_depth.generator.pth` |
| S | 8 at 96 × 96 (masked reconstruction) | `models/mae_slab_native96.pth` (backbone only: its reconstruction head is refitted for 200 steps on training tiles, backbone frozen) |
| X | 32 at 192 × 192 | `models/mae_crossres_xyz.generator.pth` |
| U-1 | 32 at 192 × 192 (the trilinear input head) | none needed |
| U-2 | 32 at 192 × 192 (the learned input head) | `models/upsampler_learned_xy2_d4.pth` |

A test that hasn't trained yet prints `not trained yet`.

In [ ]:
# setup: random monitor crops, model loading, and the 3-column layer figure
import os
import sys
import types
from pathlib import Path

REPO = Path("/vesuvius")
os.chdir(REPO)
sys.path[:0] = [str(REPO), str(REPO / "crossres")]

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from mae_pretrain_crossres import CrossResMAE, PairedSegment, build_config
from mae_pretrain_nnunet import _apply_mask, _make_spatial_mask
from pairs import PAIRS
from utils.config import Config
from utils.model import create_model
from utils.upsampler import load_upsampler

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEED = None      # an int repeats the same crops
CTX, DEPTH = 96, 8
MODELS = REPO / "models"
RNG = np.random.default_rng(SEED)


def data_cfg():
    cfg = Config()
    cfg.data.depth, cfg.data.context_size = DEPTH, CTX
    cfg.data.train_d_start, cfg.data.train_d_end = 8, 20
    return cfg


def segments(plan):
    built = [p for p in PAIRS if p["role"] == "train" and (REPO / "crossres/pairs" / plan / str(p["zid"]) / "meta.json").exists()]
    return [PairedSegment(p, data_cfg(), "global", plan) for p in built]


SEGMENTS = {plan: segments(plan) for plan in ("depth", "xyz") if (REPO / "crossres/pairs" / plan).is_dir()}


def random_crop(plan):
    """a random 96 x 96 crop of a random monitor tile: input (1,1,8,96,96), truth (1,1,32,..), valid (1,1,1,..)."""
    while True:
        seg = SEGMENTS[plan][int(RNG.integers(len(SEGMENTS[plan])))]
        tile = int(RNG.choice(seg.tiles["monitor"]))
        start = int(RNG.choice(seg.starts))
        window, target_window = seg._windows(start)
        s = seg.scale
        y, x = (int(v) for v in RNG.integers(0, seg.input.shape[-1] - CTX + 1, size=2))
        raw = seg.target[tile, target_window, s * y:s * (y + CTX), s * x:s * (x + CTX)]
        valid = seg.valid[tile, s * y:s * (y + CTX), s * x:s * (x + CTX)] & (raw > 0).all(axis=0)
        if valid.mean() >= 0.9:
            break
    x_in = seg._norm(seg.input[tile, window, y:y + CTX, x:x + CTX].astype(np.float32))
    z0 = seg.meta["z_range"][0] + start
    print(f"crop: {seg.name} monitor tile {tile}, y={y} x={x}, slices {z0}-{z0 + DEPTH - 1}")
    to = lambda a: torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32))[None, None].to(DEV)
    return to(x_in), to(seg.lut[raw]), to(valid)[:, :, None], seg


def build(depth, ctx, scale=1, factor=1):
    args = types.SimpleNamespace(fiber_coordinate_branch=False, data_norm_mode="global",
                                 depth=depth, d_start=8, d_end=20, ctx=ctx)
    backbone, _ = create_model(build_config(args))
    return CrossResMAE(backbone, depth, scale=scale, depth_factor=factor).to(DEV)


def load_full(path):
    checkpoint = torch.load(path, map_location=DEV, weights_only=False)
    model = build(checkpoint["depth"], checkpoint["ctx"], checkpoint["xy_scale"], checkpoint["depth_factor"])
    model.load_state_dict(checkpoint["state_dict"])
    return model.eval()


def refit_recon_head(model, steps=200, batch=16):
    """backbone frozen; fit the zero-initialised reconstruction head on training tiles only."""
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    opt = torch.optim.Adam(model.recon_head.parameters(), lr=1e-3)
    rng = np.random.default_rng(1)
    model.train()
    for _ in range(steps):
        seg = SEGMENTS["depth"][int(rng.integers(len(SEGMENTS["depth"])))]
        x = seg.sample("train", batch, rng)[0].to(DEV)
        mask = _make_spatial_mask(batch, CTX, 1, 4, 0.65, DEV, rng).expand(batch, 1, DEPTH, CTX, CTX).clone()
        mask[: batch // 2, :, 3:5] = 1.0
        recon, _ = model(_apply_mask(x, mask))
        loss = ((recon - x) ** 2 * mask).sum() / mask.sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model.eval()


def _np(a):
    return a.detach().float().cpu().numpy()


def layer_figure(title, truth, middle, real_rows, hidden_rows=(), valid=None):
    """truth, middle: (L, H, W). real_rows get a green frame, hidden_rows a red one."""
    truth, middle = _np(truth), _np(middle)
    ok = _np(valid).astype(bool) if valid is not None else np.ones(truth.shape[1:], bool)
    lo, hi = np.percentile(truth[:, ok], [1, 99])
    scaled = lambda a: np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
    layers = truth.shape[0]
    fig, axes = plt.subplots(layers, 3, figsize=(7.5, 2.5 * layers), squeeze=False)
    for i in range(layers):
        is_real, is_hidden = i in real_rows, i in hidden_rows
        overlay = np.dstack([scaled(middle[i]), scaled(truth[i]), np.zeros_like(truth[i])])
        tag = "REAL" if is_real else ("HIDDEN, predicted" if is_hidden else "pred")
        err = float(np.mean((middle[i][ok] - truth[i][ok]) ** 2))
        panels = [(truth[i], f"truth {i}"), (middle[i], f"{tag} {i}"), (overlay, f"overlay {i}  MSE {err:.4f}")]
        for c, (image, label) in enumerate(panels):
            ax = axes[i, c]
            ax.imshow(scaled(image) if image.ndim == 2 else image, cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(label, fontsize=8)
            if c == 1 and (is_real or is_hidden):
                for spine in ax.spines.values():
                    spine.set_edgecolor("lime" if is_real else "red")
                    spine.set_linewidth(4)
    fig.suptitle(title, fontsize=10, y=1.0)
    plt.tight_layout()
    plt.show()


def interleave(pred, x, factor, scale):
    """the predicted layers, with each real 9.36 um slice placed at the second of its `factor` target layers."""
    real = F.interpolate(x, scale_factor=(1, scale, scale), mode="nearest")[0, 0]
    stack = pred[0, 0].clone()
    rows = [k * factor + 1 for k in range(x.shape[2])]
    stack[rows] = real
    return stack, set(rows)


def sr_test(name, pred, x, t, v, factor, scale):
    stack, rows = interleave(pred, x, factor, scale)
    total = float(((pred - t) ** 2 * v).sum() / (v.expand_as(t).sum()))
    tri = F.interpolate(x, scale_factor=(factor, scale, scale), mode="trilinear", align_corners=False)
    base = float(((tri - t) ** 2 * v).sum() / (v.expand_as(t).sum()))
    layer_figure(f"{name}: all-layer MSE vs 2.4 um  prediction {total:.5f}  trilinear {base:.5f}  "
                 f"({100 * (1 - total / base):+.1f}%)\nmiddle: green frame = real 9.36 um slice, others predicted",
                 t[0, 0], stack, rows, valid=v[0, 0, 0])


print(f"device {DEV}; monitor tiles: " + ", ".join(f"{plan} {sum(len(s.tiles['monitor']) for s in segs)}"
                                                   for plan, segs in SEGMENTS.items()))

In [ ]:
# D: 32 layers at 96 x 96 (4x depth) vs the 2.4 um truth
path = MODELS / "mae_crossres_depth.generator.pth"
if not path.is_file():
    print(f"D: not trained yet ({path})")
else:
    model = load_full(path)
    x, t, v, _ = random_crop("depth")
    with torch.no_grad():
        _, pred = model(x, x)
    sr_test("D", pred, x, t, v, factor=4, scale=1)
    del model
    torch.cuda.empty_cache()

Model parameters (nnunet3d_lcndz): 7,522,618
X: not trained yet (/vesuvius/models/mae_crossres_xyz.generator.pth)


In [ ]:
# S: 8 layers at 96 x 96, masked reconstruction vs the real 9.36 um slices
path = MODELS / "mae_slab_native96.pth"
if not path.is_file():
    print(f"S: not trained yet ({path})")
else:
    model = build(DEPTH, CTX)
    model.backbone.load_state_dict(torch.load(path, map_location=DEV, weights_only=True))
    model = refit_recon_head(model)
    x, _, _, _ = random_crop("depth")
    mask = _make_spatial_mask(1, CTX, 1, 4, 0.65, DEV, RNG).expand(1, 1, DEPTH, CTX, CTX).clone()
    hidden_start = int(RNG.integers(0, DEPTH - 1))
    mask[:, :, hidden_start:hidden_start + 2] = 1.0
    with torch.no_grad():
        recon, _ = model(_apply_mask(x, mask))
    # visible voxels stay real; hidden columns and slices show the prediction
    shown = torch.where(mask > 0, recon, x)
    hidden = {hidden_start, hidden_start + 1}
    err = float(((recon - x) ** 2 * mask).sum() / mask.sum())
    layer_figure(f"S: MSE on hidden voxels {err:.5f}\nmiddle: green frame = real data in the visible 35% of columns "
                 f"(the rest predicted), red frame = whole slice hidden", x[0, 0], shown[0, 0],
                 set(range(DEPTH)) - hidden, hidden)
    del model
    torch.cuda.empty_cache()

Model parameters (nnunet3d_lcndz): 7,522,618


/usr/local/lib/python3.10/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [ ]:
# X: 32 layers at 192 x 192 (4x depth, 2x x/y) vs the 2.4 um truth
path = MODELS / "mae_crossres_xyz.generator.pth"
if not path.is_file():
    print(f"X: not trained yet ({path})")
else:
    model = load_full(path)
    x, t, v, _ = random_crop("xyz")
    with torch.no_grad():
        _, pred = model(x, x)
    sr_test("X", pred, x, t, v, factor=4, scale=2)
    del model
    torch.cuda.empty_cache()

U-1: MAE not trained yet (/vesuvius/models/mae_upsampled_trilinear_native96.full.pth)
U-2: upsampler not trained yet (/vesuvius/models/upsampler_learned_xy2_d4.pth)


In [ ]:
# U-1: the trilinear input head (v8-in style), 32 layers at 192 x 192 vs the 2.4 um truth
up = load_upsampler("trilinear").to(DEV).eval()
x, t, v, _ = random_crop("xyz")
with torch.no_grad():
    pred = up(x)
sr_test("U-1 input head (trilinear)", pred, x, t, v, factor=up.depth_factor, scale=up.scale)

,prediction,trilinear,linear 3x3x3,reconstruction
D,0.01540,0.01663,0.01372,NaN
S,NaN,NaN,NaN,0.01015
U-1,0.01701,0.01701,NaN,NaN


In [ ]:
# U-2: the learned input head, 32 layers at 192 x 192 vs the 2.4 um truth (the title also gives trilinear's MSE)
path = MODELS / "upsampler_learned_xy2_d4.pth"
if not path.is_file():
    print(f"U-2: not trained yet ({path})")
else:
    up = load_upsampler(str(path)).to(DEV).eval()
    x, t, v, _ = random_crop("xyz")
    with torch.no_grad():
        pred = up(x)
    sr_test("U-2 input head (learned)", pred, x, t, v, factor=up.depth_factor, scale=up.scale)